# Chagas Drug Discovery | Análise Exploratória de Dados
**Alvo:** Cruzipain (Cruzain) | CHEMBL3563 | *Trypanosoma cruzi*  
**Dataset:** 716 entradas brutas baixadas do ChEMBL (IC50, nM, relação =, organismo *T. cruzi*)  
**Objetivo:** Limpar o dataset, classificar inibidores por potência e entender a distribuição química dos compostos.

## 0. Imports e configuração

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

os.makedirs('../data/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

## 1. Carregamento do dataset
Dataset baixado diretamente do ChEMBL com os filtros: IC50, nM, relação `=`, organismo *Trypanosoma cruzi*.  
O separador do arquivo ChEMBL é `;` então é obrigatório passar `sep=';'`.

In [2]:
df_raw = pd.read_csv('../data/raw/chembl_cruzipain.csv', sep=';')
print(f'Shape bruto: {df_raw.shape}')
print(f'Colunas: {df_raw.shape[1]}')
df_raw.head(3)

Shape bruto: (716, 48)
Colunas: 48


,Molecule ChEMBL ID,Molecule Name,Molecule Max Phase,Molecular Weight,#RO5 Violations,AlogP,Compound Key,Smiles,Standard Type,Standard Relation,...,Document ChEMBL ID,Source ID,Source Description,Document Journal,Document Year,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value
0,CHEMBL541696,NaN,NaN,340.17,0.0,2.77,1,Nc1ncccc1C(=O)OCC(=O)Nc1cc(Cl)ccc1Cl,IC50,'=',...,CHEMBL1151964,1,Scientific Literature,J Med Chem,2009,NaN,NaN,NaN,NaN,116.0
1,CHEMBL550751,NaN,NaN,346.35,0.0,2.96,"22, SMDC-281566",Nc1ccccc1-c1nnc(C(=O)Nc2cc(-c3ccccc3)[nH]n2)o1,IC50,'=',...,CHEMBL1151964,1,Scientific Literature,J Med Chem,2009,NaN,NaN,NaN,NaN,0.8
2,CHEMBL549616,NaN,NaN,383.46,0.0,3.43,SMDC-256202,Nc1ccccc1-c1nnc(C(=O)NCc2csc(-c3cccs3)n2)o1,IC50,'=',...,CHEMBL1151964,1,Scientific Literature,J Med Chem,2009,NaN,NaN,NaN,NaN,2.0


In [3]:
# Das 48 colunas do ChEMBL, selecionar apenas as relevantes para o projeto
df = df_raw[[
    'Molecule ChEMBL ID',
    'Smiles',
    'Standard Value',
    'Data Validity Comment',
    'pChEMBL Value',
    'Document Year',
    'AlogP',
    'Molecular Weight'
]].copy()

df.columns = ['molecule_chembl_id', 'smiles', 'IC50_nM',
              'validity_comment', 'pchembl_value',
              'doc_year', 'alogp', 'mol_weight']

print(f'Shape após seleção de colunas: {df.shape}')
df.head()

Shape após seleção de colunas: (716, 8)


,molecule_chembl_id,smiles,IC50_nM,validity_comment,pchembl_value,doc_year,alogp,mol_weight
0,CHEMBL541696,Nc1ncccc1C(=O)OCC(=O)Nc1cc(Cl)ccc1Cl,116000.00,Outside typical range,NaN,2009,2.77,340.17
1,CHEMBL550751,Nc1ccccc1-c1nnc(C(=O)Nc2cc(-c3ccccc3)[nH]n2)o1,800.00,NaN,6.1,2009,2.96,346.35
2,CHEMBL549616,Nc1ccccc1-c1nnc(C(=O)NCc2csc(-c3cccs3)n2)o1,2000.00,NaN,5.7,2009,3.43,383.46
3,CHEMBL550633,Nc1ccccc1-c1nnc(C(=O)N[C@@H]2c3ccccc3C[C@@H]2O)o1,32000.00,NaN,4.5,2009,1.71,336.35
4,CHEMBL4447348,COC1=CC(=O)N(C(=O)/C=C/[C@H](Cc2ccccc2)NC(=O)[...,0.25,NaN,9.6,2019,4.66,696.93


## 2. Limpeza
Três critérios de remoção, aplicados em sequência:
1. Entradas com `Data Validity Comment = Outside typical range`, são medições fora do intervalo experimental confiável.
2. Nulos em IC50 ou SMILES linhas sem dado, sem molécula
3. IC50 > 100.000 nM que é biologicamente irrelevante para drug discovery

In [4]:
print(f'Entradas brutas: {len(df)}')
print(f"'Outside typical range': {(df['validity_comment'] == 'Outside typical range').sum()}")

df = df[df['validity_comment'] != 'Outside typical range']
df['IC50_nM'] = pd.to_numeric(df['IC50_nM'], errors='coerce')
df = df.dropna(subset=['IC50_nM', 'smiles'])
df = df[df['IC50_nM'] <= 100000]

print(f'Após limpeza: {len(df)} entradas')

Entradas brutas: 716
'Outside typical range': 37
Após limpeza: 679 entradas


## 3. Tratamento de duplicatas | Utilizamos mediana por molécula
A mesma molécula pode aparecer em múltiplos ensaios com IC50 diferentes que é uma variação normal de laboratório para laboratório.  
Decisão: Vamos consolidar usando a **mediana do IC50** por molécula, que acredito ser a medida mais robusta.

In [5]:
print(f'Moléculas únicas antes da consolidação: {df["molecule_chembl_id"].nunique()}')
print(f'Entradas duplicadas: {df["molecule_chembl_id"].duplicated().sum()}')

df_clean = df.groupby('molecule_chembl_id').agg(
    smiles        = ('smiles',        'first'),
    IC50_nM       = ('IC50_nM',       'median'),
    pchembl_value = ('pchembl_value', 'median'),
    doc_year      = ('doc_year',      'first'),
    alogp         = ('alogp',         'first'),
    mol_weight    = ('mol_weight',    'first')
).reset_index()

print(f'Dataset após consolidação por mediana: {len(df_clean)} inibidores únicos')

Moléculas únicas antes da consolidação: 511
Entradas duplicadas: 168
Dataset após consolidação por mediana: 511 inibidores únicos


## 4. Classificação de potência | classes de IC50
Replica exatamente o critério do artigo 
- **Classe 0** (potente): IC50 ≤ Q25  
- **Classe 1** (moderado): Q25 < IC50 ≤ Q50  
- **Classe 2** (fraco): IC50 > Q50

Onde Qs são os quantis

In [6]:
q25 = df_clean['IC50_nM'].quantile(0.25)
q50 = df_clean['IC50_nM'].quantile(0.50)

print(f'Threshold Classe 0/1 (Q25): {q25:.1f} nM')
print(f'Threshold Classe 1/2 (Q50): {q50:.1f} nM')
print()
print(f'Classe 0 (potente):  IC50 <= {q25:.0f} nM')
print(f'Classe 1 (moderado): {q25:.0f} < IC50 <= {q50:.0f} nM')
print(f'Classe 2 (fraco):    IC50 > {q50:.0f} nM')

def assign_class(ic50):
    if ic50 <= q25:   return 0
    elif ic50 <= q50: return 1
    else:             return 2

df_clean['potency_class'] = df_clean['IC50_nM'].apply(assign_class)
print()
print(df_clean['potency_class'].value_counts().sort_index())

Threshold Classe 0/1 (Q25): 85.8 nM
Threshold Classe 1/2 (Q50): 2800.0 nM

Classe 0 (potente):  IC50 <= 86 nM
Classe 1 (moderado): 86 < IC50 <= 2800 nM
Classe 2 (fraco):    IC50 > 2800 nM

potency_class
0    128
1    130
2    253
Name: count, dtype: int64


# 5. Remoção de moléculas para validação final
Uma molécula de cada classe é removida aleatoriamente (`random_state=42`) e salva em CSV separado.

In [7]:
hidden = pd.concat([
    df_clean[df_clean['potency_class'] == cls].sample(1, random_state=42)
    for cls in [0, 1, 2]
]).reset_index(drop=True)

hidden.to_csv('../data/processed/validation_molecules.csv', index=False)
print(hidden[['molecule_chembl_id', 'IC50_nM', 'potency_class']])

df_clean = df_clean[~df_clean['molecule_chembl_id'].isin(hidden['molecule_chembl_id'])]
print(f'\nDataset após remoção: {len(df_clean)} inibidores')

  molecule_chembl_id   IC50_nM  potency_class
0      CHEMBL4447348      0.25              0
1      CHEMBL3628940    124.00              1
2       CHEMBL566706  30000.00              2

Dataset após remoção: 508 inibidores


## 6. Salvar dataset final

In [11]:
df_clean.to_csv('../data/raw/cruzipain_final.csv', index=False)
print(f'Dataset salvo: {len(df_clean)} inibidores únicos')
print(f'Arquivo: data/raw/cruzipain_final.csv')
df_clean.describe()

Dataset salvo: 508 inibidores únicos
Arquivo: data/raw/cruzipain_final.csv


,IC50_nM,pchembl_value,doc_year,alogp,mol_weight,potency_class
count,508.000000,508.000000,508.000000,505.000000,508.000000,508.000000
mean,12999.280781,6.015659,2012.527559,3.794079,412.054724,1.246063
std,21855.480039,1.525225,6.421520,1.530154,143.685415,0.828775
min,0.081000,4.000000,2002.000000,-1.460000,154.210000,0.000000
25%,87.922500,4.835000,2009.000000,2.700000,311.362500,0.750000
50%,2800.000000,5.550000,2010.000000,3.560000,365.325000,1.000000
75%,14700.000000,7.092500,2018.000000,4.750000,473.560000,2.000000
max,100000.000000,10.090000,2025.000000,9.750000,906.930000,2.000000


## 7. Resultados da Análise

**Dataset:**
- 716 entradas brutas do ChEMBL (CHEMBL3563 - Cruzipain, *T. cruzi*)
- 37 entradas removidas por `Outside typical range`
- 180 duplicatas consolidadas pela mediana do IC50
- 3 Moléculas ocultas
- Dataset final: 508 inibidores únicos

**Moléculas ocultas:**
- Ocultamos uma molécula aleatória de cada classe, para realizar um teste no site ao final.